# Model 2: YOLOv8s-seg (Instance Segmentation)\n\nYOLOv8 small variant for instance segmentation, pre-trained on COCO.\n\n**Key improvements over baseline 0.3 mAP:**\n- Letterbox resize (preserves aspect ratio) instead of forced square\n- 150 epochs with early stopping (not 3)\n- Full augmentation suite: mosaic, mixup, copy-paste, HSV, flip, rotation\n- Cosine LR schedule with warmup\n- Instance segmentation (polygon masks) instead of detection-only\n- Label smoothing for noisy TACO labels

In [ ]:
import torch
import os

# Verify GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

# =========================
# PATHS
# =========================

BASE_DIR = r"C:\Users\User\Desktop\Ipynb"
DATASET_DIR = os.path.join(BASE_DIR, "archive", "dataset_v2")
DATASET_YAML = os.path.join(DATASET_DIR, "dataset.yaml")

print(f"\nDataset YAML: {DATASET_YAML}")
print(f"Exists: {os.path.exists(DATASET_YAML)}")

## Quick Data Sanity Check\n\nVerify dataset structure before training.

In [ ]:
# =========================
# VERIFY DATASET STRUCTURE
# =========================

for split in ["train", "val", "test"]:
    img_dir = os.path.join(DATASET_DIR, "images", split)
    lbl_dir = os.path.join(DATASET_DIR, "labels", split)
    
    img_files = set(os.path.splitext(f)[0] for f in os.listdir(img_dir) if not f.startswith('.'))
    lbl_files = set(os.path.splitext(f)[0] for f in os.listdir(lbl_dir) if not f.startswith('.'))
    
    matched = img_files & lbl_files
    img_only = img_files - lbl_files
    lbl_only = lbl_files - img_files
    
    print(f"{split}: {len(img_files)} images, {len(lbl_files)} labels, "
          f"{len(matched)} matched, {len(img_only)} img-only, {len(lbl_only)} lbl-only")

# Check a sample label
sample_lbl = os.path.join(DATASET_DIR, "labels", "train", os.listdir(os.path.join(DATASET_DIR, "labels", "train"))[0])
with open(sample_lbl) as f:
    lines = f.readlines()
print(f"\nSample label ({os.path.basename(sample_lbl)}):")
print(f"  Objects: {len(lines)}")
if lines:
    parts = lines[0].strip().split()
    print(f"  First object: class={parts[0]}, polygon_coords={len(parts)-1} values")

## Train YOLOv8s-seg\n\nTraining with aggressive augmentation and proper hyperparameters.\n\n| Parameter | Value | Why |\n|-----------|-------|-----|\n| imgsz | 1024 | Large objects in TACO, letterbox preserves aspect ratio |\n| batch | 4 | Safe for 8GB VRAM at 1024px |\n| epochs | 150 | Enough to converge with early stopping |\n| mosaic | 1.0 | Creates synthetic crowded scenes from 4 images |\n| copy_paste | 0.1 | Copies objects between images - great for small datasets |\n| patience | 30 | Stop if no improvement for 30 epochs |

In [ ]:
from ultralytics import YOLO

# Load YOLOv8s-seg pretrained on COCO
model = YOLO("yolov8s-seg.pt")

# Train with optimized settings
results = model.train(
    data=DATASET_YAML,
    epochs=150,
    imgsz=1024,
    batch=4,
    device=0,
    
    # ---- Augmentation (critical for 1500 images) ----
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.1,
    erasing=0.1,
    
    # ---- Optimizer ----
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=5,
    warmup_momentum=0.5,
    
    # ---- Scheduler & Regularization ----
    cos_lr=True,
    label_smoothing=0.1,
    
    # ---- Loss weights ----
    box=7.5,
    cls=1.0,
    dfl=1.5,
    
    # ---- Segmentation specific ----
    overlap_mask=True,
    mask_ratio=4,
    
    # ---- Early stopping ----
    patience=30,
    
    # ---- Logging ----
    project=os.path.join(BASE_DIR, "runs", "yolov8_seg"),
    name="taco_v2",
    save=True,
    save_period=25,
    plots=True,
    verbose=True,
)

print("Training complete!")

## Validate on Test Set

In [ ]:
# =========================
# EVALUATE ON TEST SET
# =========================

# Load best model
best_model_path = os.path.join(BASE_DIR, "runs", "yolov8_seg", "taco_v2", "weights", "best.pt")
best_model = YOLO(best_model_path)

# Run validation on test split
test_metrics = best_model.val(
    data=DATASET_YAML,
    split="test",
    imgsz=1024,
    batch=4,
    device=0,
    plots=True,
    save_json=True,
    verbose=True,
)

print("\n" + "=" * 50)
print("YOLOV8s-SEG TEST RESULTS")
print("=" * 50)
print(f"Box mAP50:        {test_metrics.box.map50:.4f}")
print(f"Box mAP50-95:     {test_metrics.box.map:.4f}")
print(f"Mask mAP50:       {test_metrics.seg.map50:.4f}")
print(f"Mask mAP50-95:    {test_metrics.seg.map:.4f}")
print("=" * 50)

# Per-class results
print("\nPer-class Box mAP50:")
for i, name in enumerate(test_metrics.names.values()):
    if i < len(test_metrics.box.maps):
        print(f"  {name}: {test_metrics.box.maps[i]:.4f}")

## Counting Evaluation & Inference Speed

In [ ]:
import json
import time
import numpy as np
from collections import defaultdict

# =========================
# COUNTING EVALUATION
# =========================

# Load test ground truth
with open(os.path.join(DATASET_DIR, "test_annotations.json")) as f:
    test_data = json.load(f)

gt_counts = defaultdict(int)
for ann in test_data["annotations"]:
    gt_counts[ann["image_id"]] += 1

img_id_by_name = {img["file_name"]: img["id"] for img in test_data["images"]}

# Run inference on test images
test_image_dir = os.path.join(DATASET_DIR, "images", "test")
test_files = sorted([f for f in os.listdir(test_image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

count_errors = []
inference_times = []
per_image_results = []

for fname in test_files:
    img_path = os.path.join(test_image_dir, fname)
    img_id = img_id_by_name.get(fname, -1)
    gt_count = gt_counts.get(img_id, 0)
    
    # Measure inference time
    start = time.time()
    preds = best_model(img_path, imgsz=1024, device=0, verbose=False)
    elapsed = (time.time() - start) * 1000
    inference_times.append(elapsed)
    
    pred_count = len(preds[0].boxes) if preds[0].boxes is not None else 0
    error = abs(pred_count - gt_count)
    count_errors.append(error)
    
    per_image_results.append({
        "file": fname,
        "gt_count": gt_count,
        "pred_count": pred_count,
        "count_error": error,
        "inference_ms": elapsed,
    })

within_1 = sum(1 for e in count_errors if e <= 1) / len(count_errors) * 100
within_3 = sum(1 for e in count_errors if e <= 3) / len(count_errors) * 100

print("=" * 50)
print("YOLOV8s-SEG - COUNTING & SPEED")
print("=" * 50)
print(f"Counting MAE:          {np.mean(count_errors):.2f}")
print(f"Counting within +/-1:  {within_1:.1f}%")
print(f"Counting within +/-3:  {within_3:.1f}%")
print(f"Avg inference time:    {np.mean(inference_times):.1f} ms")
print(f"Median inference time: {np.median(inference_times):.1f} ms")
print("=" * 50)

# Save results
output_dir = os.path.join(BASE_DIR, "runs", "yolov8_seg", "taco_v2")
results_data = {
    "metrics": {
        "box_map50": float(test_metrics.box.map50),
        "box_map50_95": float(test_metrics.box.map),
        "mask_map50": float(test_metrics.seg.map50),
        "mask_map50_95": float(test_metrics.seg.map),
        "counting_mae": float(np.mean(count_errors)),
        "counting_within_1": float(within_1),
        "counting_within_3": float(within_3),
        "avg_inference_ms": float(np.mean(inference_times)),
    },
    "per_image": per_image_results,
}

with open(os.path.join(output_dir, "yolov8_seg_results.json"), "w") as f:
    json.dump(results_data, f, indent=2)
print(f"Results saved to: {output_dir}/yolov8_seg_results.json")

## Visualize Predictions

In [ ]:
import matplotlib.pyplot as plt
import cv2

# =========================
# VISUALIZE PREDICTIONS
# =========================

# Select 6 diverse images
sample_files = test_files[:6] if len(test_files) >= 6 else test_files

fig, axes = plt.subplots(2, 3, figsize=(20, 14))
axes = axes.flatten()

for idx, fname in enumerate(sample_files):
    img_path = os.path.join(test_image_dir, fname)
    
    preds = best_model(img_path, imgsz=1024, device=0, verbose=False)
    result = preds[0]
    
    # Plot with masks and boxes
    annotated = result.plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    
    img_id = img_id_by_name.get(fname, -1)
    gt_count = gt_counts.get(img_id, 0)
    pred_count = len(result.boxes) if result.boxes is not None else 0
    
    axes[idx].imshow(annotated_rgb)
    axes[idx].set_title(f"GT: {gt_count} | Pred: {pred_count}", fontsize=12)
    axes[idx].axis("off")

plt.suptitle("YOLOv8s-seg Predictions on Test Set", fontsize=16)
plt.tight_layout()

viz_path = os.path.join(output_dir, "yolov8_seg_predictions.png")
plt.savefig(viz_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to: {viz_path}")